# 05 - Modelagem Baseline

Este notebook inicia a etapa de Machine Learning a partir da base analítica e da estratégia de split definidas anteriormente.

Os objetivos desta etapa são:

- reconstruir os conjuntos de treino, validação e teste;
- estabelecer um baseline ingênuo de classificação;
- preparar o pipeline de preprocessing sem vazamento de informação;
- treinar e avaliar o primeiro modelo supervisionado.

O teste temporal de 2024 permanecerá reservado e não será utilizado durante o desenvolvimento ou seleção dos modelos.

Como as principais features possuem granularidade contextual, as previsões devem ser interpretadas como estimativas da probabilidade de alfabetização de um aluno a partir das características socioeconômicas, educacionais e territoriais disponíveis.

# 1. Configuração
Nesta etapa são importadas as bibliotecas utilizadas no notebook

## Bibliotecas

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType
)

from pyspark.ml import Pipeline

from pyspark.ml.feature import (
    Imputer,
    StringIndexer,
    OneHotEncoder,
    VectorAssembler,
    StandardScaler,
)

from pyspark.ml.classification import LogisticRegression

from pyspark.ml.evaluation import BinaryClassificationEvaluator

from pyspark.ml.classification import DecisionTreeClassifier

from pyspark.ml.classification import RandomForestClassifier

## Carregamento das Bases

In [0]:
df_gold = spark.table("workspace.gold.ml_alfabetizacao_aluno")

df_split = spark.table("workspace.gold.ml_split_modelagem")

# 2. Reconstrução dos Conjuntos de Modelagem

A atribuição dos splits é associada novamente à base individual utilizando `ano` e `id_municipio`.

Dessa forma, todos os alunos pertencentes a um mesmo município permanecem no mesmo conjunto experimental.

In [0]:
df_modelagem = df_gold.join(df_split, on=["ano", "id_municipio"], how="inner")

df_treino = df_modelagem.filter(F.col("split") == "treino")
df_validacao = df_modelagem.filter(F.col("split") == "validacao")
df_teste = df_modelagem.filter(F.col("split") == "teste")

In [0]:
display(
    df_modelagem.groupBy("split")
    .agg(
        F.count("*").alias("qtd_alunos"),
        F.countDistinct("id_municipio").alias("qtd_municipios"),
        F.round(F.avg(F.col("alfabetizado").cast("double")) * 100, 2).alias(
            "taxa_alfabetizacao"
        ),
    )
    .orderBy("split")
)

# 3. Definição das Features

A primeira versão dos modelos utiliza apenas variáveis previamente selecionadas como candidatas.

Identificadores individuais e territoriais são preservados para rastreabilidade, mas não são utilizados diretamente como features.

PIB e população em valores brutos também não entram inicialmente no modelo, sendo utilizadas suas versões logarítmicas.

In [0]:
target = "alfabetizado"

features_categoricas = ["rede"]

features_numericas = [
    "inse_2021",
    "censo_pct_agua_potavel",
    "censo_pct_energia_rede_publica",
    "censo_pct_esgoto_rede_publica",
    "censo_pct_biblioteca_sala_leitura",
    "censo_pct_laboratorio_informatica",
    "censo_pct_internet_aprendizagem",
    "censo_docentes_por_100_alunos",
    "censo_turmas_por_100_alunos",
    "log_pib_per_capita",
    "log_populacao",
]

features_modelo = features_categoricas + features_numericas

# 4. Baseline Ingênuo

Como a classe majoritária corresponde aos alunos alfabetizados, é utilizado como referência um classificador ingênuo que prevê `alfabetizado = 1` para todos os
registros.

Esse baseline representa o desempenho mínimo que os modelos supervisionados devem superar.

Além da acurácia, é especialmente importante observar o desempenho da classe `0`, correspondente aos alunos não alfabetizados.

In [0]:
df_baseline_validacao = df_validacao.withColumn("prediction", F.lit(1.0))

display(
    df_baseline_validacao.groupBy("alfabetizado", "prediction")
    .count()
    .orderBy("alfabetizado", "prediction")
)

In [0]:
baseline_metricas = df_baseline_validacao.agg(
    F.count("*").alias("total"),
    F.sum(F.when(F.col("alfabetizado") == F.col("prediction"), 1).otherwise(0)).alias(
        "acertos"
    ),
    F.sum(
        F.when((F.col("alfabetizado") == 0) & (F.col("prediction") == 0), 1).otherwise(
            0
        )
    ).alias("acertos_classe_0"),
    F.sum(F.when(F.col("alfabetizado") == 0, 1).otherwise(0)).alias("total_classe_0"),
)

display(
    baseline_metricas.select(
        F.round(F.col("acertos") / F.col("total"), 4).alias("accuracy"),
        F.round(F.col("acertos_classe_0") / F.col("total_classe_0"), 4).alias(
            "recall_classe_0"
        ),
    )
)

##Referência de Desempenho

O baseline ingênuo fornece uma referência inicial para avaliação dos modelos supervisionados.

Como a classe majoritária representa aproximadamente 58% da validação, a acurácia isolada não será suficiente para avaliar a qualidade dos modelos.

As comparações posteriores considerarão também Precision, Recall, F1-score, ROC-AUC e matriz de confusão.

Será dada atenção especial ao desempenho da classe `0`, correspondente aos alunos não alfabetizados, por representar o grupo de maior interesse para identificação de risco educacional.

# 5. Pipeline de Preprocessing

O tratamento das features é realizado dentro de um pipeline de Machine Learning.

Essa abordagem garante que os parâmetros de preprocessing sejam ajustados exclusivamente sobre o conjunto de treino e posteriormente aplicados aos conjuntos de validação e teste.

O pipeline inclui:

- imputação pela mediana das features numéricas;
- indexação e codificação da variável categórica `rede`;
- consolidação das features em um vetor;
- padronização das variáveis;
- Logistic Regression.

Dessa forma, evita-se vazamento de informação entre os conjuntos de modelagem.

In [0]:
# Criando coluna label, padrão no pyspark
df_treino_ml = df_treino.withColumn("label", F.col("alfabetizado").cast("double"))
df_validacao_ml = df_validacao.withColumn("label", F.col("alfabetizado").cast("double"))

# Colunas numéricas imputadas
features_numericas_imputadas = [f"{coluna}_imp" for coluna in features_numericas]

# Imputação:
imputer = Imputer(
    inputCols=features_numericas,
    outputCols=features_numericas_imputadas,
    strategy="median",
)

# Tratamento do campo "Rede"
rede_indexer = StringIndexer(
    inputCol="rede", outputCol="rede_index", handleInvalid="keep"
)

rede_encoder = OneHotEncoder(
    inputCols=["rede_index"], outputCols=["rede_ohe"], handleInvalid="keep"
)

# Montagem das features
assembler = VectorAssembler(
    inputCols=features_numericas_imputadas + ["rede_ohe"],
    outputCol="features_raw"
)

# Padronização
scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withStd=True,
    withMean=False
)

# Regressão Logística
logistic = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",
    maxIter=100,
    regParam=0.0,
    elasticNetParam=0.0
)

Pipeline

In [0]:
pipeline_logistic = Pipeline(
    stages=[
        imputer,
        rede_indexer,
        rede_encoder,
        assembler,
        scaler,
        logistic
    ]
)

# 6. Logistic Regression

A Logistic Regression é utilizada como primeiro modelo supervisionado.

Por ser um modelo linear e interpretável, ela fornece uma referência importante para avaliar quanto das diferenças de alfabetização pode ser explicado por relações relativamente simples entre as características contextuais disponíveis.

O modelo é ajustado exclusivamente sobre o conjunto de treino de 2023.

### Treino

In [0]:
modelo_logistic = pipeline_logistic.fit(df_treino_ml)

pred_logistic_validacao = modelo_logistic.transform(df_validacao_ml)

### Métricas

In [0]:
evaluator_auc = BinaryClassificationEvaluator(
    labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC"
)

auc_logistic = evaluator_auc.evaluate(pred_logistic_validacao)

print(f"ROC-AUC validação: {auc_logistic:.4f}")

In [0]:
display(
    pred_logistic_validacao.groupBy("label", "prediction")
    .count()
    .orderBy("label", "prediction")
)

In [0]:
def calcular_metricas_classificacao(df_pred):
    resumo = df_pred.agg(
        F.count("*").alias("total"),
        F.sum(
            F.when((F.col("label") == 1) & (F.col("prediction") == 1), 1).otherwise(0)
        ).alias("tp"),
        F.sum(
            F.when((F.col("label") == 0) & (F.col("prediction") == 0), 1).otherwise(0)
        ).alias("tn"),
        F.sum(
            F.when((F.col("label") == 0) & (F.col("prediction") == 1), 1).otherwise(0)
        ).alias("fp"),
        F.sum(
            F.when((F.col("label") == 1) & (F.col("prediction") == 0), 1).otherwise(0)
        ).alias("fn"),
    ).collect()[0]

    tp = resumo["tp"]
    tn = resumo["tn"]
    fp = resumo["fp"]
    fn = resumo["fn"]

    accuracy = (tp + tn) / resumo["total"]

    precision_1 = tp / (tp + fp) if (tp + fp) else 0
    recall_1 = tp / (tp + fn) if (tp + fn) else 0
    f1_1 = (
        2 * precision_1 * recall_1 / (precision_1 + recall_1)
        if (precision_1 + recall_1)
        else 0
    )

    precision_0 = tn / (tn + fn) if (tn + fn) else 0
    recall_0 = tn / (tn + fp) if (tn + fp) else 0
    f1_0 = (
        2 * precision_0 * recall_0 / (precision_0 + recall_0)
        if (precision_0 + recall_0)
        else 0
    )

    return {
        "accuracy": accuracy,
        "precision_0": precision_0,
        "recall_0": recall_0,
        "f1_0": f1_0,
        "precision_1": precision_1,
        "recall_1": recall_1,
        "f1_1": f1_1,
    }

In [0]:
metricas_logistic = calcular_metricas_classificacao(pred_logistic_validacao)

for metrica, valor in metricas_logistic.items():
    print(f"{metrica}: {valor:.4f}")

print(f"roc_auc: {auc_logistic:.4f}")

### Resultado da Logistic Regression

A Logistic Regression apresentou desempenho limitado na base de validação.

A acurácia de 57,82% ficou abaixo do baseline ingênuo de 58,20%, enquanto o ROC-AUC de 0,5686 indicou capacidade discriminatória apenas moderadamente superior ao acaso.

O modelo apresentou forte tendência à classe majoritária, alcançando recall de 93,15% entre alunos alfabetizados, mas apenas 8,64% entre alunos não alfabetizados.

Os resultados indicam que relações lineares entre as features contextuais e o target são insuficientes para separar adequadamente as duas classes.

Como as análises exploratórias anteriores identificaram relações não lineares e possíveis interações entre características socioeconômicas, territoriais e educacionais, serão avaliados modelos baseados em árvores nas próximas etapas.

# 7. Decision Tree

A Decision Tree é avaliada como primeiro modelo não linear.

Diferentemente da Logistic Regression, árvores de decisão conseguem representar relações não lineares e interações entre as features sem que essas relações precisem ser especificadas previamente.

Esse comportamento é particularmente relevante neste projeto, pois as análises exploratórias indicaram relações não lineares entre alfabetização, INSE, porte municipal e características educacionais.

Nesta primeira avaliação é utilizada uma árvore relativamente simples, antes de qualquer otimização de hiperparâmetros.

In [0]:
assembler_tree = VectorAssembler(
    inputCols=features_numericas_imputadas + ["rede_ohe"],
    outputCol="features"
)

decision_tree = DecisionTreeClassifier(
    featuresCol="features",
    labelCol="label",
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",
    maxDepth=5,
    seed=42
)


pipeline_tree = Pipeline(
    stages=[
        imputer,
        rede_indexer,
        rede_encoder,
        assembler_tree,
        decision_tree
    ]
)

modelo_tree = pipeline_tree.fit(
    df_treino_ml
)


pred_tree_validacao = (
    modelo_tree
    .transform(df_validacao_ml)
)

In [0]:
auc_tree = evaluator_auc.evaluate(
    pred_tree_validacao
)

print(
    f"ROC-AUC validação: {auc_tree:.4f}"
)

In [0]:
display(
    pred_tree_validacao
    .groupBy(
        "label",
        "prediction"
    )
    .count()
    .orderBy(
        "label",
        "prediction"
    )
)

In [0]:
metricas_tree = calcular_metricas_classificacao(
    pred_tree_validacao
)

for metrica, valor in metricas_tree.items():
    print(
        f"{metrica}: {valor:.4f}"
    )

print(
    f"roc_auc: {auc_tree:.4f}"
)

In [0]:
resultados_modelos = [
    (
        "Baseline",
        0.5820,
        None,
        0.0
    ),
    (
        "Logistic Regression",
        metricas_logistic["accuracy"],
        auc_logistic,
        metricas_logistic["recall_0"]
    ),
    (
        "Decision Tree",
        metricas_tree["accuracy"],
        auc_tree,
        metricas_tree["recall_0"]
    )
]

df_resultados_modelos = spark.createDataFrame(
    resultados_modelos,
    [
        "modelo",
        "accuracy",
        "roc_auc",
        "recall_classe_0"
    ]
)

display(df_resultados_modelos)

### Resultado da Decision Tree

A Decision Tree apresentou comportamento mais equilibrado entre as classes em relação à Logistic Regression.

O recall da classe de alunos não alfabetizados aumentou de 8,64% para 40,52%, indicando maior capacidade de identificar casos da classe minoritária.

Entretanto, a acurácia caiu para 56,91% e o ROC-AUC para 0,5359. O resultado indica que a árvore simples possui capacidade limitada de discriminação entre as classes, apesar de reduzir a forte concentração de previsões na classe majoritária observada na Logistic Regression.

O próximo passo será avaliar um modelo de ensemble baseado em múltiplas árvores, buscando capturar relações não lineares de forma mais robusta.

# 8. Random Forest

A Random Forest combina múltiplas árvores de decisão para produzir previsões mais robustas e reduzir a dependência das regras aprendidas por uma única árvore.

O modelo é adequado para este problema por conseguir representar relações não lineares e interações entre características socioeconômicas, educacionais e territoriais.

Nesta primeira avaliação são utilizados hiperparâmetros moderados, sem processo de otimização, permitindo comparar o desempenho do ensemble com os modelos anteriores.

In [0]:
random_forest = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",
    numTrees=50,
    maxDepth=8,
    seed=42,
)

pipeline_rf = Pipeline(
    stages=[imputer, rede_indexer, rede_encoder, assembler_tree, random_forest]
)

modelo_rf = pipeline_rf.fit(df_treino_ml)

pred_rf_validacao = modelo_rf.transform(df_validacao_ml)

In [0]:
auc_rf = evaluator_auc.evaluate(pred_rf_validacao)

print(f"ROC-AUC validação: {auc_rf:.4f}")

In [0]:
display(
    pred_rf_validacao.groupBy("label", "prediction")
    .count()
    .orderBy("label", "prediction")
)

In [0]:
metricas_rf = calcular_metricas_classificacao(pred_rf_validacao)

for metrica, valor in metricas_rf.items():
    print(f"{metrica}: {valor:.4f}")

print(f"roc_auc: {auc_rf:.4f}")

# Resultados Consolidados

In [0]:
resultados_modelos = [
    ("Baseline", 0.5820, None, 0.0, None),
    (
        "Logistic Regression",
        metricas_logistic["accuracy"],
        auc_logistic,
        metricas_logistic["recall_0"],
        metricas_logistic["f1_0"],
    ),
    (
        "Decision Tree",
        metricas_tree["accuracy"],
        auc_tree,
        metricas_tree["recall_0"],
        metricas_tree["f1_0"],
    ),
    (
        "Random Forest",
        metricas_rf["accuracy"],
        auc_rf,
        metricas_rf["recall_0"],
        metricas_rf["f1_0"],
    ),
]

df_resultados_modelos = spark.createDataFrame(
    resultados_modelos,
    ["modelo", "accuracy", "roc_auc", "recall_classe_0", "f1_classe_0"],
)

display(df_resultados_modelos)

## 12. Conclusão da Primeira Rodada de Modelagem

Os três modelos avaliados apresentaram capacidade preditiva limitada na base de validação.

A Random Forest obteve o maior ROC-AUC (0,5769), mas sua acurácia permaneceu próxima ao baseline ingênuo. A Decision Tree apresentou maior recall da classe de alunos não alfabetizados, porém com redução da capacidade discriminatória geral.

Os resultados indicam que simplesmente aumentar a complexidade do algoritmo pode não ser suficiente para melhorar substancialmente a classificação.

Antes da otimização de hiperparâmetros ou da utilização de modelos mais complexos, será realizada uma revisão da base analítica para avaliar:

- a consistência da construção das features;
- o poder discriminatório das variáveis disponíveis;
- a limitação causada pela granularidade contextual das features;
- a existência de novas fontes de dados potencialmente relevantes.

Essa análise orientará a próxima iteração de engenharia de features.